# 🇰🇷 한국어 감성 분석 - KLUE BERT + NSMC

**목표:**
- `klue/bert-base` 모델로 NSMC(네이버 영화 리뷰) 데이터셋 학습
- 기본 학습 → 90% 이상 정확도 달성
- 버키팅(Bucketing) + 다이나믹 패딩(Dynamic Padding) 적용 후 비교

| 라벨 | 의미 |
|------|------|
| 0 | 부정 😠 |
| 1 | 긍정 😊 |

## 0. 환경 설정 및 라이브러리 설치

In [1]:
# 필요한 라이브러리 설치
!pip install transformers datasets evaluate accelerate sentencepiece -q

In [2]:
import os
import time
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from evaluate import load as load_metric

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 사용 디바이스: {device}")
if torch.cuda.is_available():
    print(f"   GPU 이름: {torch.cuda.get_device_name(0)}")

✅ 사용 디바이스: cuda
   GPU 이름: Tesla T4


## 1. 데이터셋 불러오기 - NSMC (Naver Sentiment Movie Corpus)

In [5]:
import pandas as pd
from datasets import Dataset, DatasetDict

# 원본 GitHub에서 직접 다운로드
!wget -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
!wget -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt

# TSV 로드 + null 제거
train_df = pd.read_csv('ratings_train.txt', sep='\t').dropna(subset=['document'])
test_df  = pd.read_csv('ratings_test.txt',  sep='\t').dropna(subset=['document'])

train_df['document'] = train_df['document'].astype(str)
test_df['document']  = test_df['document'].astype(str)

raw_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'test':  Dataset.from_pandas(test_df,  preserve_index=False),
})

print(raw_dataset)
print("\n📋 컬럼 확인:", raw_dataset['train'].column_names)
print("\n📝 샘플 데이터 (train 첫 5개):")
for i in range(5):
    sample = raw_dataset['train'][i]
    label_str = '긍정 😊' if sample['label'] == 1 else '부정 😠'
    print(f"  [{i}] {label_str} | {sample['document']}")

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 149995
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 49997
    })
})

📋 컬럼 확인: ['id', 'document', 'label']

📝 샘플 데이터 (train 첫 5개):
  [0] 부정 😠 | 아 더빙.. 진짜 짜증나네요 목소리
  [1] 긍정 😊 | 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
  [2] 부정 😠 | 너무재밓었다그래서보는것을추천한다
  [3] 부정 😠 | 교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
  [4] 긍정 😊 | 사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다


## 2. Train / Validation / Test 분할

In [6]:
# NSMC는 train/test만 제공 → train에서 validation 분리
# train 150,000개 중 10%를 validation으로 사용

train_val = raw_dataset['train'].train_test_split(test_size=0.1, seed=42)

dataset = DatasetDict({
    'train':      train_val['train'],   # 135,000개
    'validation': train_val['test'],    # 15,000개
    'test':       raw_dataset['test'],  # 50,000개
})

print("✅ 데이터셋 분할 완료!")
print(f"  Train:      {len(dataset['train']):>7,}개")
print(f"  Validation: {len(dataset['validation']):>7,}개")
print(f"  Test:       {len(dataset['test']):>7,}개")

# 라벨 분포 확인
import collections
train_labels = dataset['train']['label']
counter = collections.Counter(train_labels)
print(f"\n📊 Train 라벨 분포:")
print(f"  긍정(1): {counter[1]:>7,}개 ({counter[1]/len(train_labels)*100:.1f}%)")
print(f"  부정(0): {counter[0]:>7,}개 ({counter[0]/len(train_labels)*100:.1f}%)")

✅ 데이터셋 분할 완료!
  Train:      134,995개
  Validation:  15,000개
  Test:        49,997개

📊 Train 라벨 분포:
  긍정(1):  67,325개 (49.9%)
  부정(0):  67,670개 (50.1%)


## 3. 토크나이저 로드 및 전처리 (input_ids 변환)

In [7]:
# klue/bert-base 토크나이저 로드
MODEL_NAME = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"✅ 토크나이저 로드 완료: {MODEL_NAME}")
print(f"  어휘 크기: {tokenizer.vocab_size:,}")

# 토큰화 예시 확인
sample_text = "이 영화 정말 재미있어요! 강력 추천합니다."
encoded = tokenizer(sample_text)
print(f"\n📝 토큰화 예시:")
print(f"  원문: {sample_text}")
print(f"  input_ids: {encoded['input_ids']}")
print(f"  tokens: {tokenizer.convert_ids_to_tokens(encoded['input_ids'])}")

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

✅ 토크나이저 로드 완료: klue/bert-base
  어휘 크기: 32,000

📝 토큰화 예시:
  원문: 이 영화 정말 재미있어요! 강력 추천합니다.
  input_ids: [2, 1504, 3771, 3944, 6001, 10283, 5, 4817, 4635, 11800, 18, 3]
  tokens: ['[CLS]', '이', '영화', '정말', '재미있', '##어요', '!', '강력', '추천', '##합니다', '.', '[SEP]']


In [8]:

# 기본 전처리: max_length 고정 패딩

MAX_LENGTH = 128  # NSMC 리뷰 길이를 고려한 최대 길이

def tokenize_basic(batch):
    """고정 길이 패딩 (padding='max_length')"""
    # null/None 값 처리
    texts = [str(t) if t is not None else '' for t in batch['document']]
    return tokenizer(
        texts,
        truncation=True,
        padding='max_length',   # 모든 시퀀스를 MAX_LENGTH로 패딩
        max_length=MAX_LENGTH,
    )

# 기본 전처리 적용
tokenized_basic = dataset.map(tokenize_basic, batched=True)
tokenized_basic = tokenized_basic.rename_column('label', 'labels')
tokenized_basic.set_format('torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])

print("✅ 기본 전처리 완료!")
print(f"  컬럼: {tokenized_basic['train'].column_names}")
print(f"  input_ids shape (첫 샘플): {tokenized_basic['train'][0]['input_ids'].shape}")

Map:   0%|          | 0/134995 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

✅ 기본 전처리 완료!
  컬럼: ['id', 'document', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']
  input_ids shape (첫 샘플): torch.Size([128])


## 4. 모델 설정 및 학습 지표 정의

In [9]:
# 정확도 계산 함수
accuracy_metric = load_metric('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    return acc  # {'accuracy': 0.xxxx}

print("✅ 평가 지표 설정 완료: accuracy")

✅ 평가 지표 설정 완료: accuracy


## 5. 기본 학습 (Baseline Training)

첫 시도: 기본 하이퍼파라미터로 학습 → 정확도 90% 미만이면 튜닝

In [10]:
# ─────────────────────────────────────
# 기본 모델 로드
# ─────────────────────────────────────
model_basic = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,          # 긍정(1) / 부정(0) 이진 분류
    id2label={0: '부정', 1: '긍정'},
    label2id={'부정': 0, '긍정': 1},
)
print(f"✅ 모델 로드 완료: {MODEL_NAME}")
total_params = sum(p.numel() for p in model_basic.parameters())
print(f"  파라미터 수: {total_params:,}")

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ 모델 로드 완료: klue/bert-base
  파라미터 수: 110,618,882


In [11]:
# ─────────────────────────────────────
# [1차 시도] 기본 학습 설정
# ─────────────────────────────────────
OUTPUT_DIR_BASIC = './results/baseline'

training_args_basic = TrainingArguments(
    output_dir=OUTPUT_DIR_BASIC,
    eval_strategy='epoch',             # 에포크마다 평가
    save_strategy='epoch',
    learning_rate=3e-5,                # BERT fine-tuning 권장값
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    warmup_ratio=0.1,                  # 전체 스텝의 10% warm-up
    logging_steps=200,
    fp16=torch.cuda.is_available(),    # GPU 있으면 FP16 사용
    report_to='none',                  # wandb 등 외부 로깅 비활성화
)

trainer_basic = Trainer(
    model=model_basic,
    args=training_args_basic,
    train_dataset=tokenized_basic['train'],
    eval_dataset=tokenized_basic['validation'],
    compute_metrics=compute_metrics,
)

print("🚀 기본 학습 시작 (3 에포크, lr=3e-5)...")
start_time = time.time()
trainer_basic.train()
basic_time = time.time() - start_time
print(f"⏱️  학습 시간: {basic_time/60:.1f}분")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚀 기본 학습 시작 (3 에포크, lr=3e-5)...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.253961,0.249864,0.900667
2,0.170687,0.250043,0.907133
3,0.111826,0.324945,0.906867


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

⏱️  학습 시간: 44.0분


In [12]:
# ─────────────────────────────────────
# 기본 학습 결과 평가
# ─────────────────────────────────────
eval_basic = trainer_basic.evaluate(tokenized_basic['validation'])
basic_acc = eval_basic['eval_accuracy']

print("\n📊 [기본 학습] Validation 결과:")
print(f"  정확도 (Accuracy): {basic_acc*100:.2f}%")
print(f"  Loss: {eval_basic['eval_loss']:.4f}")

if basic_acc >= 0.90:
    print("\n✅ 목표 달성! 정확도 90% 이상입니다. 추가 튜닝 불필요.")
else:
    print(f"\n⚠️  정확도 {basic_acc*100:.2f}% → 90% 미만. 아래 셀에서 튜닝 진행합니다.")

Training Loss,Validation Loss,Epoch,Accuracy
0.111826,0.250043,3,0.907133



📊 [기본 학습] Validation 결과:
  정확도 (Accuracy): 90.71%
  Loss: 0.2500

✅ 목표 달성! 정확도 90% 이상입니다. 추가 튜닝 불필요.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# [2차 시도] 정확도 90% 미만일 경우: 하이퍼파라미터 튜닝
#   - 에포크 증가: 3 → 5
#   - 학습률 조정: 3e-5 → 5e-5
#   - warm-up 비율 증가: 0.1 → 0.15
#   - 배치 크기 감소 (더 세밀한 업데이트): 32 → 16
# ─────────────────────────────────────────────────────────────────────────

if basic_acc < 0.90:
    print("🔧 하이퍼파라미터 튜닝 시작...")

    # 메모리 정리
    del model_basic
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    model_tuned = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={0: '부정', 1: '긍정'},
        label2id={'부정': 0, '긍정': 1},
    )

    training_args_tuned = TrainingArguments(
        output_dir='./results/tuned',
        eval_strategy='epoch',
        save_strategy='epoch',
        learning_rate=5e-5,            # ↑ 학습률 상향
        per_device_train_batch_size=16, # ↓ 배치 크기 축소 (더 촘촘한 업데이트)
        per_device_eval_batch_size=64,
        num_train_epochs=5,            # ↑ 에포크 증가
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        greater_is_better=True,
        warmup_ratio=0.15,             # ↑ warm-up 비율 상향
        logging_steps=200,
        fp16=torch.cuda.is_available(),
        report_to='none',
    )

    trainer_tuned = Trainer(
        model=model_tuned,
        args=training_args_tuned,
        train_dataset=tokenized_basic['train'],
        eval_dataset=tokenized_basic['validation'],
        compute_metrics=compute_metrics,
    )

    print("🚀 튜닝 학습 시작 (5 에포크, lr=5e-5)...")
    start_time = time.time()
    trainer_tuned.train()
    tuned_time = time.time() - start_time

    eval_tuned = trainer_tuned.evaluate(tokenized_basic['validation'])
    tuned_acc = eval_tuned['eval_accuracy']

    print(f"\n📊 [튜닝 학습] Validation 결과:")
    print(f"  정확도: {tuned_acc*100:.2f}% (기존 {basic_acc*100:.2f}% → +{(tuned_acc-basic_acc)*100:.2f}%)")
    print(f"  학습 시간: {tuned_time/60:.1f}분")

    if tuned_acc >= 0.90:
        print("\n✅ 목표 달성! 튜닝 후 정확도 90% 이상 달성!")
    else:
        print("\n⚠️  추가 에포크 또는 다른 하이퍼파라미터 조합을 시도해보세요.")
else:
    print("✅ 기본 학습으로 이미 90% 이상 달성. 튜닝 건너뜀.")

In [14]:
# Test 데이터셋으로 최종 평가
# 90% 이상 달성한 모델로 테스트
best_trainer = trainer_tuned if basic_acc < 0.90 else trainer_basic

test_results_basic = best_trainer.evaluate(tokenized_basic['test'])
print("📊 [최종 테스트] Test 결과:")
print(f"  정확도: {test_results_basic['eval_accuracy']*100:.2f}%")
print(f"  Loss:   {test_results_basic['eval_loss']:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy
0.111826,0.251447,3,0.904734


📊 [최종 테스트] Test 결과:
  정확도: 90.47%
  Loss:   0.2514


---
## 6. 버키팅(Bucketing) + 다이나믹 패딩(Dynamic Padding) 학습

### 개념 정리
| 방법 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **고정 패딩** | 모든 시퀀스를 동일 길이로 패딩 | 구현 간단 | 짧은 문장에 불필요한 패딩 → 연산 낭비 |
| **다이나믹 패딩** | 배치 내 최장 시퀀스 길이에 맞춰 패딩 | 불필요한 연산 감소 | 배치마다 길이 다름 |
| **버키팅** | 비슷한 길이끼리 같은 배치로 묶기 | 패딩 최소화 + 속도 향상 | 데이터 순서 뒤섞임 |

In [15]:
# ─────────────────────────────────────────────────────────────
# 버키팅 + 다이나믹 패딩 전처리
# padding=False → 패딩 없이 토큰화만 수행
# DataCollatorWithPadding이 배치 단위로 다이나믹 패딩 적용
# ─────────────────────────────────────────────────────────────

def tokenize_dynamic(batch):
    """패딩 없이 토큰화만 (다이나믹 패딩용)"""
    texts = [str(t) if t is not None else '' for t in batch['document']]
    return tokenizer(
        texts,
        truncation=True,
        padding=False,   # ← 핵심: 패딩 없이 토큰 ID만 생성
        max_length=MAX_LENGTH,
    )

tokenized_dynamic = dataset.map(tokenize_dynamic, batched=True)
tokenized_dynamic = tokenized_dynamic.rename_column('label', 'labels')
tokenized_dynamic.set_format('torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])

print("✅ 다이나믹 패딩용 전처리 완료!")
print("  (패딩 없이 토큰화만 수행됨)")

# 배치 내 다이나믹 패딩을 담당하는 Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ DataCollatorWithPadding 준비 완료!")

Map:   0%|          | 0/134995 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

✅ 다이나믹 패딩용 전처리 완료!
  (패딩 없이 토큰화만 수행됨)
✅ DataCollatorWithPadding 준비 완료!


In [17]:
# ─────────────────────────────────────────────────────────────
# 버키팅(Bucketing) 구현
# : 비슷한 시퀀스 길이끼리 그룹핑 → 각 배치의 패딩 최소화
# ─────────────────────────────────────────────────────────────
from torch.utils.data import Sampler
import math

class BucketSampler(Sampler):
    """
    버킷 샘플러:
    1. 전체 데이터를 시퀀스 길이 기준으로 정렬
    2. 연속된 인덱스끼리 같은 배치로 묶음 (길이 비슷)
    3. 배치 단위로 셔플 (배치 내부는 유사한 길이 유지)
    """
    def __init__(self, dataset, batch_size, shuffle=True, seed=42):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        
        # 각 샘플의 input_ids 길이 계산
        lengths = [len(dataset[i]['input_ids']) for i in range(len(dataset))]
        
        # 길이 기준으로 인덱스 정렬 (짧은 → 긴 순)
        self.sorted_indices = sorted(range(len(lengths)), key=lambda i: lengths[i])
        
        # 배치 크기만큼 묶음
        self.batches = [
            self.sorted_indices[i:i + batch_size]
            for i in range(0, len(self.sorted_indices), batch_size)
        ]
    
    def __iter__(self):
        if self.shuffle:
            # 배치 단위 셔플 (각 배치 내부는 유사한 길이 유지)
            rng = np.random.default_rng(self.seed)
            batch_order = rng.permutation(len(self.batches)).tolist()
            indices = []
            for b in batch_order:
                indices.extend(self.batches[b])
        else:
            indices = [idx for batch in self.batches for idx in batch]
        return iter(indices)
    
    def __len__(self):
        return len(self.sorted_indices)

BATCH_SIZE = 32
bucket_sampler = BucketSampler(
    tokenized_dynamic['train'],
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

print(f"✅ BucketSampler 생성 완료!")
print(f"  총 배치 수: {len(bucket_sampler.batches):,}개")
print(f"  배치 크기: {BATCH_SIZE}")

✅ BucketSampler 생성 완료!
  총 배치 수: 4,219개
  배치 크기: 32


In [20]:
# ─────────────────────────────────────────────────────────────
# 버키팅 + 다이나믹 패딩 Trainer 설정
# ─────────────────────────────────────────────────────────────

# 메모리 정리
import gc
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

model_bucketing = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: '부정', 1: '긍정'},
    label2id={'부정': 0, '긍정': 1},
)

training_args_bucketing = TrainingArguments(
    output_dir='./results/bucketing',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    warmup_ratio=0.1,
    logging_steps=200,
    fp16=torch.cuda.is_available(),
    report_to='none',
    # 버키팅 사용 시: group_by_length=True 로도 유사 효과 가능
    # group_by_length=True,  ← HF Trainer 내장 버키팅 옵션
)

class BucketingTrainer(Trainer):
    """BucketSampler를 사용하는 커스텀 Trainer"""
    def __init__(self, bucket_sampler, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._bucket_sampler = bucket_sampler
    
    def _get_train_sampler(self, *args, **kwargs):
        return self._bucket_sampler

trainer_bucketing = BucketingTrainer(
    bucket_sampler=bucket_sampler,
    model=model_bucketing,
    args=training_args_bucketing,
    train_dataset=tokenized_dynamic['train'],
    eval_dataset=tokenized_dynamic['validation'],
    data_collator=data_collator,           # ← 다이나믹 패딩 핵심!
    compute_metrics=compute_metrics,
)

print("🚀 버키팅 + 다이나믹 패딩 학습 시작...")
start_time = time.time()
trainer_bucketing.train()
bucketing_time = time.time() - start_time
print(f"⏱️  학습 시간: {bucketing_time/60:.1f}분")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_rat

🚀 버키팅 + 다이나믹 패딩 학습 시작...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.256532,0.249165,0.902333
2,0.162064,0.282283,0.906133
3,0.101923,0.346807,0.906267


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

⏱️  학습 시간: 21.8분


In [21]:
# 버키팅 모델 평가
eval_bucketing = trainer_bucketing.evaluate(tokenized_dynamic['validation'])
bucketing_acc = eval_bucketing['eval_accuracy']

print("📊 [버키팅 + 다이나믹 패딩] Validation 결과:")
print(f"  정확도: {bucketing_acc*100:.2f}%")
print(f"  Loss:   {eval_bucketing['eval_loss']:.4f}")

# Test 평가
test_bucketing = trainer_bucketing.evaluate(tokenized_dynamic['test'])
print(f"\n📊 [버키팅 + 다이나믹 패딩] Test 결과:")
print(f"  정확도: {test_bucketing['eval_accuracy']*100:.2f}%")

Training Loss,Validation Loss,Epoch,Accuracy
0.101923,0.346807,3,0.906267


📊 [버키팅 + 다이나믹 패딩] Validation 결과:
  정확도: 90.63%
  Loss:   0.3468


Training Loss,Validation Loss,Epoch,Accuracy
0.101923,0.350637,3,0.905234



📊 [버키팅 + 다이나믹 패딩] Test 결과:
  정확도: 90.52%


## 7. 결과 비교

In [22]:
# ─────────────────────────────────────────────────────────────
# 최종 결과 비교표
# ─────────────────────────────────────────────────────────────

basic_val_acc = eval_basic['eval_accuracy']
basic_test_acc = test_results_basic['eval_accuracy']
bucket_val_acc = eval_bucketing['eval_accuracy']
bucket_test_acc = test_bucketing['eval_accuracy']

print("=" * 65)
print(" " * 15 + "📊 최종 결과 비교")
print("=" * 65)
print(f"{'방법':<25} {'Val 정확도':>12} {'Test 정확도':>12} {'학습 시간':>12}")
print("-" * 65)
print(f"{'고정 패딩 (기본)':<25} {basic_val_acc*100:>11.2f}% {basic_test_acc*100:>11.2f}% {basic_time/60:>10.1f}분")
print(f"{'버키팅 + 다이나믹 패딩':<25} {bucket_val_acc*100:>11.2f}% {bucket_test_acc*100:>11.2f}% {bucketing_time/60:>10.1f}분")
print("=" * 65)

# 속도 비교
time_diff = basic_time - bucketing_time
if time_diff > 0:
    print(f"\n⚡ 버키팅이 {time_diff/60:.1f}분 더 빠름 ({time_diff/basic_time*100:.1f}% 속도 향상)")
else:
    print(f"\n📝 기본 방법이 {-time_diff/60:.1f}분 더 빠름")

acc_diff = bucket_val_acc - basic_val_acc
print(f"📈 정확도 차이 (Validation): {acc_diff*100:+.2f}%p")
print()
print("💡 버키팅 + 다이나믹 패딩의 주요 효과:")
print("  - 불필요한 패딩 토큰 감소 → 연산량 절약")
print("  - 배치 내 유사한 길이의 문장 → 효율적인 GPU 활용")
print("  - 긴 문장/짧은 문장이 섞인 데이터에서 특히 효과적")

               📊 최종 결과 비교
방법                             Val 정확도     Test 정확도        학습 시간
-----------------------------------------------------------------
고정 패딩 (기본)                      90.71%       90.47%       44.0분
버키팅 + 다이나믹 패딩                   90.63%       90.52%       21.8분

⚡ 버키팅이 22.3분 더 빠름 (50.6% 속도 향상)
📈 정확도 차이 (Validation): -0.09%p

💡 버키팅 + 다이나믹 패딩의 주요 효과:
  - 불필요한 패딩 토큰 감소 → 연산량 절약
  - 배치 내 유사한 길이의 문장 → 효율적인 GPU 활용
  - 긴 문장/짧은 문장이 섞인 데이터에서 특히 효과적


import torch
print(torch.cuda.is_available())       # True면 GPU 있음
print(torch.cuda.get_device_name(0))   # GPU 이름 출력

In [23]:
# 실제 문장으로 감성 분석 테스트
def predict_sentiment(text, model, tokenizer, device):
    """단일 문장 감성 분석 추론"""
    model.eval()
    model.to(device)
    
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    pred_label = np.argmax(probs)
    
    return {
        'text': text,
        'label': '긍정 😊' if pred_label == 1 else '부정 😠',
        'confidence': f'{probs[pred_label]*100:.1f}%',
        'prob_pos': f'{probs[1]*100:.1f}%',
        'prob_neg': f'{probs[0]*100:.1f}%',
    }

# 테스트 문장들
test_sentences = [
    "이 영화 진짜 최고입니다! 배우들 연기도 훌륭하고 스토리도 완벽해요.",
    "시간 낭비였어요. 스토리도 없고 지루하기만 했습니다.",
    "배우들 연기력은 훌륭하지만 스토리가 너무 진부해서 아쉬워요.",
    "인생 영화 추가! 보는 내내 울었습니다 ㅠㅠ 강추강추!",
    "돈 아깝다. 이걸 왜 봤지... 별점 0점 주고 싶음.",
]

# 버키팅 모델로 추론
print("🎬 감성 분석 추론 결과 (버키팅 모델)")
print("=" * 60)
for sent in test_sentences:
    result = predict_sentiment(sent, model_bucketing, tokenizer, device)
    print(f"📝 입력: {result['text'][:35]}..." if len(result['text']) > 35 else f"📝 입력: {result['text']}")
    print(f"   결과: {result['label']}  (확신도: {result['confidence']})")
    print(f"   긍정 확률: {result['prob_pos']} | 부정 확률: {result['prob_neg']}")
    print("-" * 60)

🎬 감성 분석 추론 결과 (버키팅 모델)
📝 입력: 이 영화 진짜 최고입니다! 배우들 연기도 훌륭하고 스토리도 완벽...
   결과: 긍정 😊  (확신도: 99.9%)
   긍정 확률: 99.9% | 부정 확률: 0.1%
------------------------------------------------------------
📝 입력: 시간 낭비였어요. 스토리도 없고 지루하기만 했습니다.
   결과: 부정 😠  (확신도: 100.0%)
   긍정 확률: 0.0% | 부정 확률: 100.0%
------------------------------------------------------------
📝 입력: 배우들 연기력은 훌륭하지만 스토리가 너무 진부해서 아쉬워요.
   결과: 부정 😠  (확신도: 99.0%)
   긍정 확률: 1.0% | 부정 확률: 99.0%
------------------------------------------------------------
📝 입력: 인생 영화 추가! 보는 내내 울었습니다 ㅠㅠ 강추강추!
   결과: 긍정 😊  (확신도: 99.9%)
   긍정 확률: 99.9% | 부정 확률: 0.1%
------------------------------------------------------------
📝 입력: 돈 아깝다. 이걸 왜 봤지... 별점 0점 주고 싶음.
   결과: 부정 😠  (확신도: 100.0%)
   긍정 확률: 0.0% | 부정 확률: 100.0%
------------------------------------------------------------


---
## 📚 정리

### 핵심 개념 요약

| 개념 | 설명 |
|------|------|
| **NSMC** | 네이버 영화 리뷰 감성 분석 데이터셋 (150K train, 50K test) |
| **klue/bert-base** | 한국어 전용 BERT 모델 (KorBERT 기반) |
| **고정 패딩** | `padding='max_length'`로 모든 시퀀스를 동일 길이로 패딩 |
| **다이나믹 패딩** | `DataCollatorWithPadding`이 배치마다 최적 길이로 패딩 |
| **버키팅** | 비슷한 길이의 문장끼리 묶어 배치 → 패딩 낭비 최소화 |
| **Trainer** | HuggingFace의 학습 추상화 클래스 (자동 GPU, 평가, 저장) |

### 버키팅 + 다이나믹 패딩이 유리한 경우
- 시퀀스 길이 편차가 큰 데이터셋 (짧은 문장과 긴 문장이 섞인 경우)
- 메모리/연산 자원이 제한적일 때
- 대용량 데이터셋에서 학습 속도를 높이고 싶을 때